# Core Customer Segmentation & Revenue Opportunity Execution Engine
This notebook ingests synthetic transactional logs, builds out RFM metrics, calculates structural category churn weights, and runs a mock revenue valuation simulation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Ingest and convert dates
df = pd.read_csv("synthetic_retail_transactions.csv")
df['transaction_date'] = pd.to_convert_datetime = pd.to_datetime(df['transaction_date'])
df.head()

In [ ]:
# Calculate recency, frequency, and monetary values per customer
snapshot_date = df['transaction_date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg({
    'transaction_date': lambda x: (snapshot_date - x.max()).days,
    'transaction_id': 'count',
    'basket_value_sar': 'mean'
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary_Value']

# Isolate the precise churn metrics
single_purchase_count = rfm[rfm['Frequency'] == 1].shape[0]
total_customers = rfm.shape[0]
churn_rate = (single_purchase_count / total_customers) * 100

print(f"Total Unique Profile Database: {total_customers}")
print(f"Single Purchase Trial Profiles: {single_purchase_count}")
print(f"Calculated Category Trial-to-Churn Rate: {churn_rate:.2f}%")

In [ ]:
# Analyze promotional sensitivity among highly frequent shoppers
frequent_shoppers_ids = rfm[rfm['Frequency'] > 1]['customer_id']
freq_df = df[df['customer_id'].isin(frequent_shoppers_ids)]

promo_purchase_ratio = freq_df['is_promotional'].mean() * 100
promo_revenue_contribution = (freq_df[freq_df['is_promotional'] == 1]['basket_value_sar'].sum() / freq_df['basket_value_sar'].sum()) * 100

print(f"Percentage of Frequent Shopper TXs on Promotion: {promo_purchase_ratio:.2f}%")
print(f"Percentage of Frequent Shopper Revenue Sourced via Promotions: {promo_revenue_contribution:.2f}%")

In [ ]:
# Financial model opportunity sizing visualization
plt.figure(figsize=(10, 5))
categories = ['Current Retained Base Sales', 'Identified Leaked Category Opportunity Space']
values = [df['basket_value_sar'].sum() / 1000, 20800] # Normalized units

sns.barplot(x=categories, y=values, palette=['#4A90E2', '#E2574A'])
plt.title("Valuation Matrix: Customer Category Leakage vs Current Run-Rate")
plt.ylabel("Value in Thousands (Normalized Scale)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()